# InfraVision: train the person detector on POP (Kaggle version)

Same training as the Colab notebook: fine-tune the FLIR thermal YOLO11s model (or `yolo11s.pt`
if the zip has no `thermal_best.pt`) on POP
drone thermal images at 1024 px. Kaggle gives about **30 free GPU hours per week**, and one
session can run for up to **12 hours**, which is enough for the whole training.

**One-time setup**
1. Kaggle account → **Settings → Phone verification** (needed for GPU and internet).
2. **Datasets → New Dataset** → upload `pop_yolo_colab.zip` (Kaggle unzips it). Any name is fine.
   Build the zip on the laptop with `convert_pop.py`, then `make_pop_zip.py` (both in `human_detection/datasets/`).
3. *Optional, only if Colab already trained some epochs:* download
   `My Drive/Infravision/runs/pop_1024/weights/last.pt` and upload it as a second dataset.
   The notebook will continue from it instead of starting over.

**Run**
1. **Code → New Notebook → File → Import Notebook** → this `.ipynb` file.
2. Right panel: **Add Input** → your dataset(s); **Accelerator → GPU T4 x2**; **Internet → On**.
3. Click **Save Version → Save & Run All (Commit)**. It runs in the background,
   so you can close the browser. Results appear in the version's **Output** tab.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Set Accelerator -> GPU T4 x2 in the right panel."
print("GPU ready:", torch.cuda.get_device_name(0))

## 2. Install the same Ultralytics version used on the laptop

In [ ]:
import subprocess, sys

# A failed "!pip" does not stop the notebook, so check the result here.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.191"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "pip install failed. Right panel -> Internet -> On (needs a phone-verified account), then Save & Run All again."
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. Find the dataset and settings

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# Locate the unzipped dataset, wherever Kaggle mounted it
pop_dirs = [p.parent for p in INPUT.rglob("data.yaml") if (p.parent / "images" / "train").is_dir()]
assert pop_dirs, "POP dataset not found. Add your uploaded dataset with 'Add Input'."
POP_DIR = pop_dirs[0]

starts = list(INPUT.rglob("thermal_best.pt"))
# No FLIR model uploaded: start from the standard COCO model
START_WEIGHTS = starts[0] if starts else Path("yolo11s.pt")

# Optional: partial Colab run uploaded as a dataset
previous = [p for p in INPUT.rglob("last.pt")]
PREVIOUS_LAST = previous[0] if previous else None

# Kaggle inputs are read-only, so write data.yaml to the working folder
DATA_YAML = WORK / "pop_data.yaml"
DATA_YAML.write_text(
    f"path: {POP_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: person\n"
)

RUNS_DIR = WORK / "runs"
RUN_NAME = "pop_1024"
RUN_DIR = RUNS_DIR / RUN_NAME

IMGSZ = 1024     # keep small people visible
EPOCHS = 60
PATIENCE = 15    # stop early if val mAP does not improve for 15 epochs
BATCH = 8        # fits one T4 (16 GB) at 1024 px

for split in ("train", "val", "test"):
    print(f"{split:5s}: {len(list((POP_DIR / 'images' / split).iterdir()))} images")
print("Dataset       :", POP_DIR)
print("Start weights :", START_WEIGHTS)
print("Previous run  :", PREVIOUS_LAST or "none (fresh start)")

## 4. Baseline: the starting model on POP *before* training

In [ ]:
from ultralytics import YOLO

before = YOLO(str(START_WEIGHTS)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                                      device=0, plots=False, project=str(WORK / "val"), name="before")
print(f"BEFORE  precision={before.box.mp:.3f} recall={before.box.mr:.3f} "
      f"mAP50={before.box.map50:.3f} mAP50-95={before.box.map:.3f}")

## 5. Train on POP

Continues a partial Colab run if you uploaded its `last.pt`; otherwise starts from the FLIR model (or `yolo11s.pt`).
Roughly 4–5 minutes per epoch on a T4 at 1024 px.

In [ ]:
if PREVIOUS_LAST is not None:
    print("Continuing Colab run from:", PREVIOUS_LAST)
    try:
        YOLO(str(PREVIOUS_LAST)).train(
            resume=True,
            data=str(DATA_YAML),     # replace the Colab path stored in the checkpoint
            save_dir=str(RUN_DIR),   # write to Kaggle's output folder
            device=0,
            workers=2,
        )
    except AssertionError as e:
        print("Nothing to resume (training already finished):", e)
else:
    print("Starting new run from:", START_WEIGHTS)
    YOLO(str(START_WEIGHTS)).train(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        patience=PATIENCE,
        batch=BATCH,
        device=0,
        workers=2,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        # Drone images are top-down, so vertical flips are realistic too
        flipud=0.5,
    )

## 6. Evaluate on the POP test split (never used for training)

In [ ]:
best = RUN_DIR / "weights" / "best.pt"
after = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                            device=0, project=str(WORK / "val"), name="after")

print()
print(f"{'':8s} {'precision':>9s} {'recall':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
print(f"{'BEFORE':8s} {before.box.mp:9.3f} {before.box.mr:7.3f} {before.box.map50:7.3f} {before.box.map:9.3f}")
print(f"{'AFTER':8s} {after.box.mp:9.3f} {after.box.mr:7.3f} {after.box.map50:7.3f} {after.box.map:9.3f}")

# Synthetic flood test set (scripts/prepare_flood_aug.py), if the zip has it
if (POP_DIR / "images" / "test_flood").is_dir():
    flood_yaml = WORK / "pop_flood.yaml"
    flood_yaml.write_text(DATA_YAML.read_text().replace("test: images/test", "test: images/test_flood"))
    fl = YOLO(str(best)).val(data=str(flood_yaml), split="test", imgsz=IMGSZ, batch=BATCH,
                             device=0, project=str(WORK / "val"), name="after_flood")
    print(f"{'FLOOD':8s} {fl.box.mp:9.3f} {fl.box.mr:7.3f} {fl.box.map50:7.3f} {fl.box.map:9.3f}")

## 7. Training curve and example detections

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display

df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()
display(df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10))
display(Show(str(RUN_DIR / "results.png"), width=900))

YOLO(str(best)).predict(sorted((POP_DIR / "images" / "test").iterdir())[::150][:6],
                        imgsz=IMGSZ, conf=0.25, iou=0.5, save=True,
                        project=str(WORK / "pred"), name="pop", exist_ok=True)
for p in sorted((WORK / "pred" / "pop").iterdir()):
    display(Show(str(p), width=640))

## 8. Package the results

Creates `/kaggle/working/pop_1024.zip`. Download it from the notebook version's **Output** tab,
then put it in your laptop's Downloads folder.

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / "pop_1024"), "zip", RUN_DIR)
print("Saved:", archive)